# F1 Predictor

This notebook upgrades the original F1 race predictor from mostly grid/qualifying features into:

**Driver × Car × Track × Environment → Pace → Reliability → Strategy/Raceability → Monte Carlo probabilities**

It contains:
- corner geometry and per-corner braking/apex/exit/aero features,
- car capability and track-demand vectors,
- track–car compatibility and nearest-track transfer,
- teammate-adjusted driver skill,
- weather/wind/temperature features,
- tyre degradation and warm-up,
- practice pace corrected for track evolution,
- dirty-air/overtaking/reliability/strategy modules,
- LambdaMART race ranking,
- separate DNF modeling,
- calibrated race simulation.

**Leakage rule:** at every prediction snapshot, only information known at that timestamp is legal.

In [ ]:
from pathlib import Path
import base64, sys, os, json, shutil, subprocess

WORK = Path('/kaggle/working/f1_v2') if Path('/kaggle/working').exists() else Path('./f1_v2_work')
WORK.mkdir(parents=True, exist_ok=True)

payload = {"f1pred/legacy/features.py": "...", "f1pred/legacy/leakage.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgcGFuZGFzIGFzIHBkCgpTVEFHRV9PUkRFUj17IlBSRV9XRUVLRU5EIjowLCJQT1NUX0ZQMSI6MSwiUE9TVF9GUDIiOjIsIlBPU1RfRlAzIjozLCJQT1NUX1FVQUxJIjo0LCJSQUNFX1NUQVJUIjo1LCJQT1NUX1JBQ0UiOjZ9CgpGRUFUVVJFX1NUQUdFPXsKICAgICJoaXN0b3JpY2FsX2Zvcm0iOiJQUkVfV0VFS0VORCIsInRyYWNrX2dlb21ldHJ5IjoiUFJFX1dFRUtFTkQiLCJ0cmFja19zaW1pbGFyaXR5IjoiUFJFX1dFRUtFTkQiLCJhcmNoaXZlZF93ZWF0aGVyIjoiUFJFX1dFRUtFTkQiLAogICAgImZwMSI6IlBPU1RfRlAxIiwiZnAyIjoiUE9TVF9GUDIiLCJmcDMiOiJQT1NUX0ZQMyIsInF1YWxpZnlpbmciOiJQT1NUX1FVQUxJIiwiZ3JpZCI6IlBPU1RfUVVBTEkiLAogICAgInJhY2Vfd2VhdGhlciI6IlBPU1RfUkFDRSIsInJhY2Vfc3RpbnRzIjoiUE9TVF9SQUNFIiwicGl0X3N0b3BzIjoiUE9TVF9SQUNFIiwicmFjZV9sYXBzIjoiUE9TVF9SQUNFIiwicmVzdWx0IjoiUE9TVF9SQUNFIiwKfQoKZGVmIGFsbG93ZWQoZmVhdHVyZV9mYW1pbHk6IHN0ciwgcHJlZGljdGlvbl9zdGFnZTogc3RyKSAtPiBib29sOgogICAgbmVlZD1GRUFUVVJFX1NUQUdFLmdldChmZWF0dXJlX2ZhbWlseSwiUE9TVF9SQUNFIikKICAgIHJldHVybiBTVEFHRV9PUkRFUltuZWVkXSA8PSBTVEFHRV9PUkRFUltwcmVkaWN0aW9uX3N0YWdlXQoKZGVmIGFzc2VydF9ub19mdXR1cmVfZmVhdHVyZXMoZmVhdHVyZV9mYW1pbGllcywgcHJlZGljdGlvbl9zdGFnZSk6CiAgICBiYWQ9W2YgZm9yIGYgaW4gZmVhdHVyZV9mYW1pbGllcyBpZiBub3QgYWxsb3dlZChmLHByZWRpY3Rpb25fc3RhZ2UpXQogICAgaWYgYmFkOiByYWlzZSBWYWx1ZUVycm9yKGYiTGVha2FnZSByaXNrIGF0IHtwcmVkaWN0aW9uX3N0YWdlfToge2JhZH0iKQogICAgcmV0dXJuIFRydWUKCmRlZiBhc3NlcnRfdGltZXN0YW1wX2N1dG9mZihkZjogcGQuRGF0YUZyYW1lLCBmZWF0dXJlX3RpbWVfY29sOiBzdHIsIHByZWRpY3Rpb25fdGltZV9jb2w6IHN0cik6CiAgICBmPXBkLnRvX2RhdGV0aW1lKGRmW2ZlYXR1cmVfdGltZV9jb2xdLHV0Yz1UcnVlLGVycm9ycz0iY29lcmNlIik7IHA9cGQudG9fZGF0ZXRpbWUoZGZbcHJlZGljdGlvbl90aW1lX2NvbF0sdXRjPVRydWUsZXJyb3JzPSJjb2VyY2UiKQogICAgYmFkPShmPnApJmYubm90bmEoKSZwLm5vdG5hKCkKICAgIGlmIGJhZC5hbnkoKTogcmFpc2UgVmFsdWVFcnJvcihmIntpbnQoYmFkLnN1bSgpKX0gZmVhdHVyZSByb3dzIG9jY3VyIGFmdGVyIHByZWRpY3Rpb24gdGltZXN0YW1wIikKICAgIHJldHVybiBUcnVlCg==", "f1pred/legacy/data_sources.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQganNvbiwgdGltZSwgaGFzaGxpYgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB1cmxsaWIucGFyc2UgaW1wb3J0IHVybGVuY29kZSwgcXVvdGUKaW1wb3J0IHBhbmRhcyBhcyBwZAoKT1BFTkYxPSJodHRwczovL2FwaS5vcGVuZjEub3JnL3YxIgpPUEVOX01FVEVPX1BSRVZJT1VTPSJodHRwczovL3ByZXZpb3VzLXJ1bnMtYXBpLm9wZW4tbWV0ZW8uY29tL3YxL2ZvcmVjYXN0IgpPUEVOX01FVEVPX0hJU1Q9Imh0dHBzOi8vaGlzdG9yaWNhbC1mb3JlY2FzdC1hcGkub3Blbi1tZXRlby5jb20vdjEvZm9yZWNhc3QiClRSQUNJTkdfUkFXPSJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vVHJhY2luZ0luc2lnaHRzLzIwMjYvbWFpbiIKCgpkZWYgb3BlbmYxX3VybChlbmRwb2ludDogc3RyLCAqKnBhcmFtcykgLT4gc3RyOgogICAgcmV0dXJuIGYie09QRU5GMX0ve3F1b3RlKGVuZHBvaW50LnN0cmlwKCcvJykpfT97dXJsZW5jb2RlKHBhcmFtcyxkb3NlcT1UcnVlKX0iCgpkZWYgdHJhY2luZ191cmwoZXZlbnQ6IHN0ciwgc2Vzc2lvbjogc3RyLCBmaWxlbmFtZTogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gZiJ7VFJBQ0lOR19SQVd9L3txdW90ZShldmVudCl9L3txdW90ZShzZXNzaW9uKX0ve3F1b3RlKGZpbGVuYW1lKX0iCgpkZWYgb3Blbm1ldGVvX3ByZXZpb3VzX3VybChsYXQ6IGZsb2F0LCBsb246IGZsb2F0LCBzdGFydF9kYXRlOiBzdHIsIGVuZF9kYXRlOiBzdHIsIGhvdXJseTogbGlzdFtzdHJdLCBwYXN0X2RheXM6IGludHxOb25lPU5vbmUpIC0+IHN0cjoKICAgIHBhcmFtcz17ImxhdGl0dWRlIjpsYXQsImxvbmdpdHVkZSI6bG9uLCJzdGFydF9kYXRlIjpzdGFydF9kYXRlLCJlbmRfZGF0ZSI6ZW5kX2RhdGUsImhvdXJseSI6IiwiLmpvaW4oaG91cmx5KSwidGltZXpvbmUiOiJVVEMifQogICAgaWYgcGFzdF9kYXlzIGlzIG5vdCBOb25lOiBwYXJhbXNbInBhc3RfZGF5cyJdPXBhc3RfZGF5cwogICAgcmV0dXJuIE9QRU5fTUVURU9fUFJFVklPVVMrIj8iK3VybGVuY29kZShwYXJhbXMpCgpkZWYgbm9ybWFsaXplX29wZW5mMV9jYXJfZGF0YShkZjogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBkPWRmLmNvcHkoKTsgcmVuPXsiZGF0ZSI6InRpbWVzdGFtcCIsInNwZWVkIjoic3BlZWQiLCJ0aHJvdHRsZSI6InRocm90dGxlIiwiYnJha2UiOiJicmFrZSIsInJwbSI6InJwbSIsIm5fZ2VhciI6ImdlYXIiLCJkcnMiOiJkcnMifTsgZD1kLnJlbmFtZShjb2x1bW5zPXJlbikKICAgIGZvciBjIGluIFsic3BlZWQiLCJ0aHJvdHRsZSIsImJyYWtlIiwicnBtIiwiZ2VhciIsImRycyJdOgogICAgICAgIGlmIGMgaW4gZDogZFtjXT1wZC50b19udW1lcmljKGRbY10sZXJyb3JzPSJjb2VyY2UiKQogICAgaWYgInRpbWVzdGFtcCIgaW4gZDogZFsidGltZXN0YW1wIl09cGQudG9fZGF0ZXRpbWUoZFsidGltZXN0YW1wIl0sdXRjPVRydWUsZXJyb3JzPSJjb2VyY2UiKQogICAgcmV0dXJuIGQKCmRlZiBub3JtYWxpemVfb3BlbmYxX3dlYXRoZXIoZGY6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgZD1kZi5jb3B5KCkucmVuYW1lKGNvbHVtbnM9eyJkYXRlIjoidGltZXN0YW1wIiwiYWlyX3RlbXBlcmF0dXJlIjoiYWlyX3RlbXAiLCJ0cmFja190ZW1wZXJhdHVyZSI6InRyYWNrX3RlbXAiLCJ3aW5kX3NwZWVkIjoid2luZF9zcGVlZCIsIndpbmRfZGlyZWN0aW9uIjoid2luZF9kaXJlY3Rpb24iLCJyYWluZmFsbCI6InJhaW5mYWxsIiwiaHVtaWRpdHkiOiJodW1pZGl0eSIsInByZXNzdXJlIjoicHJlc3N1cmUifSkKICAgIGlmICJ0aW1lc3RhbXAiIGluIGQ6IGQudGltZXN0YW1wPXBkLnRvX2RhdGV0aW1lKGQudGltZXN0YW1wLHV0Yz1UcnVlLGVycm9ycz0iY29lcmNlIikKICAgIHJldHVybiBkCgpkZWYgbm9ybWFsaXplX3RyYWNpbmdfdGVsZW1ldHJ5KGRmOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgIGQ9ZGYuY29weSgpOyBsb3dlcj17YzpjLnN0cmlwKCkubG93ZXIoKS5yZXBsYWNlKCIgIiwiXyIpIGZvciBjIGluIGQuY29sdW1uc307IGQ9ZC5yZW5hbWUoY29sdW1ucz1sb3dlcikKICAgIGFsaWFzZXM9eyJkaXN0YW5jZV9tIjoiZGlzdGFuY2UiLCJzcGVlZF9rbWgiOiJzcGVlZCIsInRocm90dGxlX3BjdCI6InRocm90dGxlIiwiYnJha2VfcGN0IjoiYnJha2UiLCJ4X20iOiJ4IiwieV9tIjoieSIsInpfbSI6InoiLCJ0aW1lIjoidGltZV9zIn0KICAgIGQ9ZC5yZW5hbWUoY29sdW1ucz17azp2IGZvciBrLHYgaW4gYWxpYXNlcy5pdGVtcygpIGlmIGsgaW4gZC5jb2x1bW5zfSkKICAgIHJldHVybiBkCgpkZWYgaHR0cF9nZXRfY2FjaGVkKHVybDogc3RyLCBjYWNoZV9kaXI6IHN0cnxQYXRoLCB0aW1lb3V0OiBpbnQ9MzAsIHJldHJpZXM6IGludD0zKSAtPiBQYXRoOgogICAgIiIiT3B0aW9uYWwgcnVudGltZSBkb3dubG9hZGVyLiBVc2VzIHJlcXVlc3RzIG9ubHkgd2hlbiBjYWxsZWQ7IGFsbCBkb3dubG9hZHMgYXJlIGNhY2hlZC4iIiIKICAgIGltcG9ydCByZXF1ZXN0cwogICAgY2FjaGU9UGF0aChjYWNoZV9kaXIpOyBjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGtleT1oYXNobGliLnNoYTI1Nih1cmwuZW5jb2RlKCkpLmhleGRpZ2VzdCgpWzoyNF07IG91dD1jYWNoZS9mIntrZXl9LmJpbiIKICAgIGlmIG91dC5leGlzdHMoKSBhbmQgb3V0LnN0YXQoKS5zdF9zaXplPjA6IHJldHVybiBvdXQKICAgIGVycj1Ob25lCiAgICBmb3IgaSBpbiByYW5nZShyZXRyaWVzKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHI9cmVxdWVzdHMuZ2V0KHVybCx0aW1lb3V0PXRpbWVvdXQpOyByLnJhaXNlX2Zvcl9zdGF0dXMoKTsgb3V0LndyaXRlX2J5dGVzKHIuY29udGVudCk7IHJldHVybiBvdXQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycj1lOyB0aW1lLnNsZWVwKDEuNSooaSsxKSkKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImRvd25sb2FkIGZhaWxlZDoge3VybH0iKSBmcm9tIGVycgo=", "f1pred/legacy/modeling.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgbWF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb24KCgpkZWYgc29mdG1heF9ieV9yYWNlKHNjb3JlcywgcmFjZV9pZHMsIHRlbXBlcmF0dXJlPTEuMCk6CiAgICBzPW5wLmFzYXJyYXkoc2NvcmVzLGZsb2F0KTsgcj1ucC5hc2FycmF5KHJhY2VfaWRzKTsgb3V0PW5wLnplcm9zKGxlbihzKSxmbG9hdCk7IHQ9bWF4KGZsb2F0KHRlbXBlcmF0dXJlKSwxZS02KQogICAgZm9yIHJpZCBpbiBwZC51bmlxdWUocik6CiAgICAgICAgaWR4PW5wLndoZXJlKHI9PXJpZClbMF07IHo9c1tpZHhdL3Q7IHo9ei1ucC5uYW5tYXgoeik7IGU9bnAuZXhwKHopOyBvdXRbaWR4XT1lL2Uuc3VtKCkKICAgIHJldHVybiBvdXQKCgpkZWYgbW9udGVfY2FybG9fcmFjZShkcml2ZXJzOiBwZC5EYXRhRnJhbWUsIG5fc2ltczogaW50PTEwMDAwLCBzZWVkOiBpbnQ9NDIsCiAgICAgICAgICAgICAgICAgICAgIHBhY2VfY29sPSJwYWNlX3Njb3JlIiwgZG5mX3Byb2JfY29sPSJkbmZfcHJvYiIsIHN0cmF0ZWd5X3NkX2NvbD0ic3RyYXRlZ3lfc2QiKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJTaW1wbGUgdHJhbnNwYXJlbnQgc2ltdWxhdG9yOiBsYXRlbnQgcGFjZSArIHN0cmF0ZWd5IG5vaXNlLCB0aGVuIGluZGVwZW5kZW50IERORiBldmVudC4iIiIKICAgIHJuZz1ucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCk7IGQ9ZHJpdmVycy5yZXNldF9pbmRleChkcm9wPVRydWUpLmNvcHkoKTsgbj1sZW4oZCkKICAgIHBhY2U9cGQudG9fbnVtZXJpYyhkW3BhY2VfY29sXSxlcnJvcnM9ImNvZXJjZSIpLmZpbGxuYSgwKS50b19udW1weShmbG9hdCkKICAgIGRuZj1wZC50b19udW1lcmljKGQuZ2V0KGRuZl9wcm9iX2NvbCwwLjA1KSxlcnJvcnM9ImNvZXJjZSIpLmZpbGxuYSgwLjA1KS5jbGlwKDAsMC45NSkudG9fbnVtcHkoZmxvYXQpCiAgICBzdHJhdD1wZC50b19udW1lcmljKGQuZ2V0KHN0cmF0ZWd5X3NkX2NvbCwwLjE1KSxlcnJvcnM9ImNvZXJjZSIpLmZpbGxuYSgwLjE1KS5jbGlwKGxvd2VyPTApLnRvX251bXB5KGZsb2F0KQogICAgd2lucz1ucC56ZXJvcyhuKTsgcG9kcz1ucC56ZXJvcyhuKTsgdG9wMTA9bnAuemVyb3Mobik7IGRuZnM9bnAuemVyb3MobikKICAgIGZvciBfIGluIHJhbmdlKGludChuX3NpbXMpKToKICAgICAgICBwZXJmPXBhY2Urcm5nLm5vcm1hbCgwLHN0cmF0LG4pCiAgICAgICAgcmV0aXJlZD1ybmcucmFuZG9tKG4pPGRuZgogICAgICAgIGRuZnMgKz0gcmV0aXJlZAogICAgICAgICMgUmV0aXJlbWVudHMgcmFua2VkIGJlaGluZCBmaW5pc2hlcnMsIHdpdGggdGhlaXIgbGF0ZW50IHBlcmZvcm1hbmNlIG9ubHkgdGllLWJyZWFraW5nLgogICAgICAgIHNjb3JlPXBlcmYtcmV0aXJlZC5hc3R5cGUoZmxvYXQpKjFlNgogICAgICAgIG9yZGVyPW5wLmFyZ3NvcnQoLXNjb3JlKQogICAgICAgIHdpbnNbb3JkZXJbMF1dKz0xOyBwb2RzW29yZGVyWzptaW4oMyxuKV1dKz0xOyB0b3AxMFtvcmRlcls6bWluKDEwLG4pXV0rPTEKICAgIG91dD1kLmNvcHkoKTsgb3V0WyJ3aW5fcHJvYmFiaWxpdHkiXT13aW5zL25fc2ltczsgb3V0WyJwb2RpdW1fcHJvYmFiaWxpdHkiXT1wb2RzL25fc2ltczsgb3V0WyJ0b3AxMF9wcm9iYWJpbGl0eSJdPXRvcDEwL25fc2ltczsgb3V0WyJkbmZfcHJvYmFiaWxpdHlfc2ltIl09ZG5mcy9uX3NpbXMKICAgIHJldHVybiBvdXQKCgpkZWYgZml0X2RuZl9jbGFzc2lmaWVyKGRmOiBwZC5EYXRhRnJhbWUsIGZlYXR1cmVzOiBsaXN0W3N0cl0sIHRhcmdldD0iZG5mIik6CiAgICBkPWRmW2ZlYXR1cmVzK1t0YXJnZXRdXS5hcHBseShwZC50b19udW1lcmljLGVycm9ycz0iY29lcmNlIikuZHJvcG5hKCkKICAgIGlmIGRbdGFyZ2V0XS5udW5pcXVlKCk8MjogcmFpc2UgVmFsdWVFcnJvcigiRE5GIHRhcmdldCByZXF1aXJlcyBib3RoIGNsYXNzZXMiKQogICAgbT1Mb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MTAwMCxjbGFzc193ZWlnaHQ9ImJhbGFuY2VkIikuZml0KGRbZmVhdHVyZXNdLGRbdGFyZ2V0XS5hc3R5cGUoaW50KSkKICAgIHJldHVybiBtCgoKZGVmIHJhY2VfcmFua2luZ19tZXRyaWNzKGRmOiBwZC5EYXRhRnJhbWUsIHNjb3JlX2NvbD0nc2NvcmUnKToKICAgIGZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IHNwZWFybWFucgogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IG5kY2dfc2NvcmUKICAgIHJvd3M9W10KICAgIGZvciBfLGcgaW4gZGYuZ3JvdXBieSgncmFjZUlkJyk6CiAgICAgICAgZz1nLnNvcnRfdmFsdWVzKHNjb3JlX2NvbCxhc2NlbmRpbmc9RmFsc2UpLmNvcHkoKTsgZ1sncHJlZF9yYW5rJ109bnAuYXJhbmdlKDEsbGVuKGcpKzEpCiAgICAgICAgaWYgJ3dpbm5lcicgaW4gZyBhbmQgZy53aW5uZXIuc3VtKCk9PTE6CiAgICAgICAgICAgIHdyPWludChnLmxvY1tnLndpbm5lcj09MSwncHJlZF9yYW5rJ10uaWxvY1swXSkKICAgICAgICBlbHNlOiB3cj1ucC5uYW4KICAgICAgICByaG89c3BlYXJtYW5yKGcucHJlZF9yYW5rLGcuZmluaXNoX3Bvc2l0aW9uLG5hbl9wb2xpY3k9J29taXQnKS5zdGF0aXN0aWMgaWYgJ2ZpbmlzaF9wb3NpdGlvbicgaW4gZyBlbHNlIG5wLm5hbgogICAgICAgIG1hZT1ucC5tZWFuKG5wLmFicyhnLnByZWRfcmFuay1nLmZpbmlzaF9wb3NpdGlvbikpIGlmICdmaW5pc2hfcG9zaXRpb24nIGluIGcgZWxzZSBucC5uYW4KICAgICAgICBpZiB7J3JlbGV2YW5jZScsc2NvcmVfY29sfS5pc3N1YnNldChnLmNvbHVtbnMpOgogICAgICAgICAgICBuZD1uZGNnX3Njb3JlKFtnLnJlbGV2YW5jZS50b19udW1weSgpXSxbZ1tzY29yZV9jb2xdLnRvX251bXB5KCldLGs9bWluKDUsbGVuKGcpKSkKICAgICAgICBlbHNlOiBuZD1ucC5uYW4KICAgICAgICByb3dzLmFwcGVuZChbd3I9PTEgaWYgbnAuaXNmaW5pdGUod3IpIGVsc2UgbnAubmFuLHdyPD0zIGlmIG5wLmlzZmluaXRlKHdyKSBlbHNlIG5wLm5hbixyaG8sbWFlLG5kXSkKICAgIGE9bnAuYXNhcnJheShyb3dzLGZsb2F0KQogICAgcmV0dXJuIHsnbl9yYWNlcyc6bGVuKGEpLCd0b3AxJzpucC5uYW5tZWFuKGFbOiwwXSksJ3dpbm5lcl90b3AzJzpucC5uYW5tZWFuKGFbOiwxXSksJ3NwZWFybWFuJzpucC5uYW5tZWFuKGFbOiwyXSksJ3JhbmtfbWFlJzpucC5uYW5tZWFuKGFbOiwzXSksJ25kY2c1JzpucC5uYW5tZWFuKGFbOiw0XSl9CgoKZGVmIGZpdF94Z2JfcmFua2VyKHRyYWluOiBwZC5EYXRhRnJhbWUsIHRlc3Q6IHBkLkRhdGFGcmFtZSwgZmVhdHVyZXM6IGxpc3Rbc3RyXSwgbGFiZWw9J3JlbGV2YW5jZScpOgogICAgaW1wb3J0IHhnYm9vc3QgYXMgeGdiCiAgICB0cj10cmFpbi5zb3J0X3ZhbHVlcyhbJ3JhY2VJZCcsJ2RyaXZlcklkJ10pLmNvcHkoKTsgdGU9dGVzdC5zb3J0X3ZhbHVlcyhbJ3JhY2VJZCcsJ2RyaXZlcklkJ10pLmNvcHkoKQogICAgWHRyPXRyW2ZlYXR1cmVzXS5hcHBseShwZC50b19udW1lcmljLGVycm9ycz0nY29lcmNlJyk7IFh0ZT10ZVtmZWF0dXJlc10uYXBwbHkocGQudG9fbnVtZXJpYyxlcnJvcnM9J2NvZXJjZScpCiAgICBtb2RlbD14Z2IuWEdCUmFua2VyKG9iamVjdGl2ZT0ncmFuazpuZGNnJyxldmFsX21ldHJpYz0nbmRjZ0A1Jyx0cmVlX21ldGhvZD0naGlzdCcsbl9lc3RpbWF0b3JzPTUwMCxtYXhfZGVwdGg9NCxsZWFybmluZ19yYXRlPS4wMzUsCiAgICAgICAgICAgICAgICAgICAgICAgIHN1YnNhbXBsZT0uOSxjb2xzYW1wbGVfYnl0cmVlPS45LHJlZ19sYW1iZGE9MixtaW5fY2hpbGRfd2VpZ2h0PTMsbGFtYmRhcmFua19wYWlyX21ldGhvZD0ndG9waycsbGFtYmRhcmFua19udW1fcGFpcl9wZXJfc2FtcGxlPTgsCiAgICAgICAgICAgICAgICAgICAgICAgIHJhbmRvbV9zdGF0ZT00MixuX2pvYnM9LTEpCiAgICBtb2RlbC5maXQoWHRyLHRyW2xhYmVsXS5hc3R5cGUoZmxvYXQpLHFpZD10ci5yYWNlSWQudG9fbnVtcHkoKSx2ZXJib3NlPUZhbHNlKQogICAgdGVbJ3Njb3JlJ109bW9kZWwucHJlZGljdChYdGUpCiAgICByZXR1cm4gbW9kZWwsdGUKCgpkZWYgdGVtcGVyYXR1cmVfY2FsaWJyYXRlKHNjb3JlcywgcmFjZV9pZHMsIHdpbm5lcnMsIGdyaWQ9Tm9uZSk6CiAgICBncmlkPW5wLmdlb21zcGFjZSguMSw2LDYwKSBpZiBncmlkIGlzIE5vbmUgZWxzZSBucC5hc2FycmF5KGdyaWQsZmxvYXQpCiAgICBiZXN0PSgxLjAsbnAuaW5mKQogICAgZm9yIHQgaW4gZ3JpZDoKICAgICAgICBwPXNvZnRtYXhfYnlfcmFjZShzY29yZXMscmFjZV9pZHMsdCk7IGxvc3Nlcz1bXQogICAgICAgIGZvciByaWQgaW4gcGQudW5pcXVlKHJhY2VfaWRzKToKICAgICAgICAgICAgaWR4PW5wLndoZXJlKG5wLmFzYXJyYXkocmFjZV9pZHMpPT1yaWQpWzBdOyB5PW5wLmFzYXJyYXkod2lubmVycylbaWR4XTsgcHA9cFtpZHhdCiAgICAgICAgICAgIGlmIHkuc3VtKCk9PTE6IGxvc3Nlcy5hcHBlbmQoLW1hdGgubG9nKG1heChwcFt5LmFzdHlwZShib29sKV1bMF0sMWUtMTIpKSkKICAgICAgICBsb3NzPW5wLm1lYW4obG9zc2VzKSBpZiBsb3NzZXMgZWxzZSBucC5pbmYKICAgICAgICBpZiBsb3NzPGJlc3RbMV06IGJlc3Q9KGZsb2F0KHQpLGZsb2F0KGxvc3MpKQogICAgcmV0dXJuIHsndGVtcGVyYXR1cmUnOmJlc3RbMF0sJ2xvZ19sb3NzJzpiZXN0WzFdfQo=", "f1pred/legacy/aggregation.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIC5waHlzaWNzX2ZlYXR1cmVzIGltcG9ydCB0cmFja19kZW1hbmRfdmVjdG9yLCBjYXJfY2FwYWJpbGl0eV92ZWN0b3IsIGNvc2luZV9jb21wYXRpYmlsaXR5CgoKZGVmIGFnZ3JlZ2F0ZV9kcml2ZXJfY29ybmVycyhjb3JuZXJzOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgIGM9Y29ybmVycy5jb3B5KCk7IGNbJ3NwZWVkX2NsYXNzJ109cGQuY3V0KHBkLnRvX251bWVyaWMoYy5hcGV4X3NwZWVkX2ttaCxlcnJvcnM9J2NvZXJjZScpLFstbnAuaW5mLDE0MCwyMjAsbnAuaW5mXSxsYWJlbHM9Wydsb3cnLCdtZWRpdW0nLCdoaWdoJ10pCiAgICByb3dzPVtdCiAgICBmb3IgZHJpdmVyLGcgaW4gYy5ncm91cGJ5KCdkcml2ZXInKToKICAgICAgICByb3c9eydkcml2ZXInOmRyaXZlciwKICAgICAgICAgICAgICdicmFraW5nX3N0cmVuZ3RoJzpwZC50b19udW1lcmljKGcuYnJha2luZ19lZmZpY2llbmN5X21zMixlcnJvcnM9J2NvZXJjZScpLm1lYW4oKSwKICAgICAgICAgICAgICd0cmFjdGlvbl9zdHJlbmd0aCc6cGQudG9fbnVtZXJpYyhnLnRyYWN0aW9uX2luZGV4X21zMixlcnJvcnM9J2NvZXJjZScpLm1lYW4oKSwKICAgICAgICAgICAgICdhZXJvX2NvbW1pdG1lbnQnOnBkLnRvX251bWVyaWMoZy5hZXJvX2NvbW1pdG1lbnQsZXJyb3JzPSdjb2VyY2UnKS5tZWFuKCksCiAgICAgICAgICAgICAnZXhpdF9hbXBsaWZpY2F0aW9uJzpwZC50b19udW1lcmljKGcuZXhpdF9hbXBsaWZpY2F0aW9uLGVycm9ycz0nY29lcmNlJykubWVhbigpfQogICAgICAgIGZvciBjbHMgaW4gWydsb3cnLCdtZWRpdW0nLCdoaWdoJ106CiAgICAgICAgICAgIHo9Z1tnLnNwZWVkX2NsYXNzPT1jbHNdCiAgICAgICAgICAgIHJvd1tmJ3tjbHN9X3NwZWVkX3BlcmZvcm1hbmNlJ109cGQudG9fbnVtZXJpYyh6LmNvcm5lcl9wZXJmb3JtYW5jZSxlcnJvcnM9J2NvZXJjZScpLm1lYW4oKSBpZiBsZW4oeikgZWxzZSBucC5uYW4KICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpCgoKZGVmIGJ1aWxkX3RyYWNrX3ZlY3Rvcihjb3JuZXJzOiBwZC5EYXRhRnJhbWUpIC0+IHBkLlNlcmllczoKICAgICMgRGVkdXBsaWNhdGUgcmVwZWF0ZWQgZHJpdmVyIG9ic2VydmF0aW9ucyBwZXIgY29ybmVyIGJlZm9yZSBjb25zdHJ1Y3RpbmcgY2lyY3VpdCBkZW1hbmQuCiAgICBjb2xzPVsnY29ybmVyJywnYXBleF9zcGVlZF9rbWgnLCdicmFraW5nX2VmZmljaWVuY3lfbXMyJywndHJhY3Rpb25faW5kZXhfbXMyJywnYWVyb19jb21taXRtZW50JywnZXhpdF9hbXBsaWZpY2F0aW9uJ10KICAgIHQ9Y29ybmVyc1tjb2xzXS5ncm91cGJ5KCdjb3JuZXInLGFzX2luZGV4PUZhbHNlKS5tZWRpYW4obnVtZXJpY19vbmx5PVRydWUpCiAgICByZXR1cm4gdHJhY2tfZGVtYW5kX3ZlY3Rvcih0KQoKCmRlZiBjb21wYXRpYmlsaXR5X2Zyb21fbmFtZWRfdmVjdG9ycyh0cmFjazogcGQuU2VyaWVzLCBjYXI6IHBkLlNlcmllcykgLT4gZmxvYXQ6CiAgICAjIE1hdGNoIHNlbWFudGljIGRpbWVuc2lvbnMgcmF0aGVyIHRoYW4gcmVseWluZyBvbiBhcmJpdHJhcnkgY29sdW1uIG9yZGVyLgogICAgcGFpcnM9WygnbG93X3NwZWVkX3NoYXJlJywnbG93X3NwZWVkJyksKCdtZWRpdW1fc3BlZWRfc2hhcmUnLCdtZWRpdW1fc3BlZWQnKSwoJ2hpZ2hfc3BlZWRfc2hhcmUnLCdoaWdoX3NwZWVkJyksKCdicmFraW5nX2RlbWFuZCcsJ2JyYWtpbmcnKSwoJ3RyYWN0aW9uX2RlbWFuZCcsJ3RyYWN0aW9uJyksKCdhZXJvX2RlbWFuZCcsJ2Flcm8nKV0KICAgIGE9W107IGI9W10KICAgIGZvciB0ayxjayBpbiBwYWlyczoKICAgICAgICBpZiB0ayBpbiB0cmFjayBhbmQgY2sgaW4gY2FyOiBhLmFwcGVuZCh0cmFja1t0a10pOyBiLmFwcGVuZChjYXJbY2tdKQogICAgcmV0dXJuIGNvc2luZV9jb21wYXRpYmlsaXR5KGEsYikK", "f1pred/legacy/session_pipeline.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gLnBoeXNpY3NfZmVhdHVyZXMgaW1wb3J0ICgKICAgIGN1cnZhdHVyZV9mcm9tX3h5LCByZXNhbXBsZV90ZWxlbWV0cnksIHNlZ21lbnRfY29ybmVycywgY29ybmVyX21ldHJpY3MsCiAgICBiYWNrd2FyZF9hc29mX3dlYXRoZXIsIHRyYWNrX2V2b2x1dGlvbl9jb3JyZWN0LCBwcmFjdGljZV9wYWNlCikKCgpkZWYgZmFzdGYxX3Nlc3Npb24oeWVhcjogaW50LCBldmVudCwgc2Vzc2lvbl9jb2RlOiBzdHIsIGNhY2hlX2Rpcjogc3RyfFBhdGgpOgogICAgaW1wb3J0IGZhc3RmMQogICAgY2FjaGU9UGF0aChjYWNoZV9kaXIpOyBjYWNoZS5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGZhc3RmMS5DYWNoZS5lbmFibGVfY2FjaGUoc3RyKGNhY2hlKSkKICAgIHM9ZmFzdGYxLmdldF9zZXNzaW9uKHllYXIsZXZlbnQsc2Vzc2lvbl9jb2RlKQogICAgcy5sb2FkKHRlbGVtZXRyeT1UcnVlLHdlYXRoZXI9VHJ1ZSxtZXNzYWdlcz1GYWxzZSkKICAgIHJldHVybiBzCgoKZGVmIGZhc3RmMV9jb3JuZXJzKHNlc3Npb24pIC0+IHBkLkRhdGFGcmFtZToKICAgIGNpPXNlc3Npb24uZ2V0X2NpcmN1aXRfaW5mbygpCiAgICBjPWNpLmNvcm5lcnMuY29weSgpCiAgICByZW49eyJOdW1iZXIiOiJjb3JuZXIiLCJEaXN0YW5jZSI6ImRpc3RhbmNlIiwiQW5nbGUiOiJhbmdsZSJ9CiAgICBjPWMucmVuYW1lKGNvbHVtbnM9e2s6diBmb3Igayx2IGluIHJlbi5pdGVtcygpIGlmIGsgaW4gYy5jb2x1bW5zfSkKICAgIGtlZXA9W3ggZm9yIHggaW4gWyJjb3JuZXIiLCJkaXN0YW5jZSIsImFuZ2xlIl0gaWYgeCBpbiBjLmNvbHVtbnNdCiAgICByZXR1cm4gY1trZWVwXS5jb3B5KCkKCgpkZWYgbGFwX3RlbGVtZXRyeV90YWJsZShsYXApIC0+IHBkLkRhdGFGcmFtZToKICAgIHRlbD1sYXAuZ2V0X3RlbGVtZXRyeSgpLmNvcHkoKQogICAgcmVuPXsiRGlzdGFuY2UiOiJkaXN0YW5jZSIsIlNwZWVkIjoic3BlZWQiLCJUaHJvdHRsZSI6InRocm90dGxlIiwiQnJha2UiOiJicmFrZSIsIlJQTSI6InJwbSIsIm5HZWFyIjoiZ2VhciIsIkRSUyI6ImRycyIsIlgiOiJ4IiwiWSI6InkiLCJaIjoieiIsIlRpbWUiOiJ0aW1lX2RlbHRhIn0KICAgIHRlbD10ZWwucmVuYW1lKGNvbHVtbnM9e2s6diBmb3Igayx2IGluIHJlbi5pdGVtcygpIGlmIGsgaW4gdGVsLmNvbHVtbnN9KQogICAgaWYgInRpbWVfZGVsdGEiIGluIHRlbDogdGVsWyJ0aW1lX3MiXT10ZWwudGltZV9kZWx0YS5kdC50b3RhbF9zZWNvbmRzKCkKICAgIGZvciBjIGluIFsiZGlzdGFuY2UiLCJzcGVlZCIsInRocm90dGxlIiwiYnJha2UiLCJycG0iLCJnZWFyIiwiZHJzIiwieCIsInkiLCJ6Il06CiAgICAgICAgaWYgYyBpbiB0ZWw6IHRlbFtjXT1wZC50b19udW1lcmljKHRlbFtjXSxlcnJvcnM9ImNvZXJjZSIpCiAgICByZXR1cm4gdGVsCgoKZGVmIGRyaXZlcl9jb3JuZXJfZmVhdHVyZXMoc2Vzc2lvbiwgZHJpdmVyLCBsYXBfc2VsZWN0b3I9ImZhc3Rlc3QiLCBkaXN0YW5jZV9zdGVwX209Mi4wKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBsYXBzPXNlc3Npb24ubGFwcy5waWNrX2RyaXZlcnMoZHJpdmVyKQogICAgaWYgbGFwcy5lbXB0eTogcmV0dXJuIHBkLkRhdGFGcmFtZSgpCiAgICBsYXA9bGFwcy5waWNrX2Zhc3Rlc3QoKSBpZiBsYXBfc2VsZWN0b3I9PSJmYXN0ZXN0IiBlbHNlIGxhcHMuaWxvY1tpbnQobGFwX3NlbGVjdG9yKV0KICAgIHRlbD1sYXBfdGVsZW1ldHJ5X3RhYmxlKGxhcCkKICAgIG5lZWQ9W2MgZm9yIGMgaW4gWyJkaXN0YW5jZSIsInNwZWVkIiwidGhyb3R0bGUiLCJicmFrZSIsIngiLCJ5Il0gaWYgYyBpbiB0ZWxdCiAgICB0ZWw9dGVsLmRyb3BuYShzdWJzZXQ9WyJkaXN0YW5jZSIsInNwZWVkIl0pCiAgICBpZiB0ZWwuZW1wdHk6IHJldHVybiBwZC5EYXRhRnJhbWUoKQogICAgdGVsPXJlc2FtcGxlX3RlbGVtZXRyeSh0ZWwsZGlzdGFuY2Vfc3RlcF9tKQogICAgaWYgeyJ4IiwieSJ9Lmlzc3Vic2V0KHRlbC5jb2x1bW5zKTogdGVsWyJjdXJ2YXR1cmUiXT1jdXJ2YXR1cmVfZnJvbV94eSh0ZWwueCx0ZWwueSkKICAgIGNvcm5lcnM9ZmFzdGYxX2Nvcm5lcnMoc2Vzc2lvbikKICAgIHNlZz1zZWdtZW50X2Nvcm5lcnModGVsLGNvcm5lcnMsYmVmb3JlX209MTIwLGFmdGVyX209MTgwKQogICAgcm93cz1bXQogICAgZGlzdGFuY2VzPWNvcm5lcnMuc29ydF92YWx1ZXMoJ2Rpc3RhbmNlJykuZGlzdGFuY2UudG9fbnVtcHkoZmxvYXQpCiAgICBsYXBfbGVuPWZsb2F0KHRlbC5kaXN0YW5jZS5tYXgoKSkKICAgIGZvciBpLChjaWQsZykgaW4gZW51bWVyYXRlKHNlZy5ncm91cGJ5KCdjb3JuZXInKSk6CiAgICAgICAgY2VudGVyPWZsb2F0KGcuY29ybmVyX2Rpc3RhbmNlLmlsb2NbMF0pOwogICAgICAgICMgZG93bnN0cmVhbSBzdHJhaWdodCBsZW5ndGggYXBwcm94aW1hdGVkIHRvIG5leHQgb2ZmaWNpYWwgY29ybmVyIG1hcmtlcgogICAgICAgIGxhdGVyPWRpc3RhbmNlc1tkaXN0YW5jZXM+Y2VudGVyXQogICAgICAgIG5leHRfZD1mbG9hdChsYXRlci5taW4oKSkgaWYgbGVuKGxhdGVyKSBlbHNlIGxhcF9sZW4KICAgICAgICBtPWNvcm5lcl9tZXRyaWNzKGcsIG1heCgwLG5leHRfZC1jZW50ZXIpKQogICAgICAgIG0udXBkYXRlKHsiZHJpdmVyIjpzdHIoZHJpdmVyKSwiY29ybmVyIjpjaWQsImNvcm5lcl9kaXN0YW5jZSI6Y2VudGVyfSkKICAgICAgICByb3dzLmFwcGVuZChtKQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzKQoKCmRlZiBzZXNzaW9uX2Nvcm5lcl9kYXRhc2V0KHNlc3Npb24sIGRyaXZlcnM9Tm9uZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgZHJpdmVycz1kcml2ZXJzIG9yIGxpc3Qoc2Vzc2lvbi5sYXBzLkRyaXZlci5kcm9wbmEoKS5hc3R5cGUoc3RyKS51bmlxdWUoKSkKICAgIGZyYW1lcz1bXQogICAgZm9yIGQgaW4gZHJpdmVyczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIHg9ZHJpdmVyX2Nvcm5lcl9mZWF0dXJlcyhzZXNzaW9uLGQpCiAgICAgICAgICAgIGlmIGxlbih4KTogZnJhbWVzLmFwcGVuZCh4KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICBvdXQ9cGQuY29uY2F0KGZyYW1lcyxpZ25vcmVfaW5kZXg9VHJ1ZSkgaWYgZnJhbWVzIGVsc2UgcGQuRGF0YUZyYW1lKCkKICAgIGlmIGxlbihvdXQpOgogICAgICAgICMgV2l0aGluLWNvcm5lciBub3JtYWxpemVkIHBlcmZvcm1hbmNlOiBsb3dlciBjb3JuZXIgdGltZSBpcyBpZGVhbCwgYnV0IHdoZW4gb25seSBtZXRyaWNzIGV4aXN0LAogICAgICAgICMgY29tYmluZSBub3JtYWxpemVkIGFwZXgrZXhpdCBzcGVlZCBhcyBhIHRyYW5zcGFyZW50IGZpcnN0IHByb3h5LgogICAgICAgIG91dFsiY29ybmVyX3BlcmZvcm1hbmNlIl09KAogICAgICAgICAgICBvdXQuZ3JvdXBieSgnY29ybmVyJylbJ2FwZXhfc3BlZWRfa21oJ10udHJhbnNmb3JtKGxhbWJkYSBzOihzLXMubWVhbigpKS8ocy5zdGQoZGRvZj0wKSBvciAxKSkgKwogICAgICAgICAgICBvdXQuZ3JvdXBieSgnY29ybmVyJylbJ2V4aXRfc3BlZWRfa21oJ10udHJhbnNmb3JtKGxhbWJkYSBzOihzLXMubWVhbigpKS8ocy5zdGQoZGRvZj0wKSBvciAxKSkKICAgICAgICApLzIKICAgIHJldHVybiBvdXQKCgpkZWYgc2Vzc2lvbl9wcmFjdGljZV9zdW1tYXJ5KHNlc3Npb24pIC0+IHBkLkRhdGFGcmFtZToKICAgIGw9c2Vzc2lvbi5sYXBzLmNvcHkoKQogICAgaWYgbC5lbXB0eTogcmV0dXJuIHBkLkRhdGFGcmFtZSgpCiAgICBkPXBkLkRhdGFGcmFtZSh7CiAgICAgICAgJ2RyaXZlcic6bC5Ecml2ZXIuYXN0eXBlKHN0ciksCiAgICAgICAgJ2xhcF90aW1lJzpsLkxhcFRpbWUuZHQudG90YWxfc2Vjb25kcygpLAogICAgICAgICdzZXNzaW9uX3NlY29uZHMnOmwuVGltZS5kdC50b3RhbF9zZWNvbmRzKCksCiAgICAgICAgJ3N0aW50JzpwZC50b19udW1lcmljKGwuU3RpbnQsZXJyb3JzPSdjb2VyY2UnKSBpZiAnU3RpbnQnIGluIGwgZWxzZSBucC5uYW4sCiAgICAgICAgJ3R5cmVfYWdlJzpwZC50b19udW1lcmljKGwuVHlyZUxpZmUsZXJyb3JzPSdjb2VyY2UnKSBpZiAnVHlyZUxpZmUnIGluIGwgZWxzZSBucC5uYW4sCiAgICAgICAgJ2NvbXBvdW5kJzpsLkNvbXBvdW5kLmFzdHlwZShzdHIpIGlmICdDb21wb3VuZCcgaW4gbCBlbHNlICcnLAogICAgfSkKICAgIGRbJ2NvcnJlY3RlZF9sYXAnXT10cmFja19ldm9sdXRpb25fY29ycmVjdChkLCdzZXNzaW9uX3NlY29uZHMnLCdsYXBfdGltZScpCiAgICByZXR1cm4gcHJhY3RpY2VfcGFjZShkLCdkcml2ZXInLCdjb3JyZWN0ZWRfbGFwJywnc3RpbnQnKQoKCmRlZiBmYXN0ZjFfd2VhdGhlcihzZXNzaW9uKSAtPiBwZC5EYXRhRnJhbWU6CiAgICB3PXNlc3Npb24ud2VhdGhlcl9kYXRhLmNvcHkoKQogICAgaWYgdy5lbXB0eTogcmV0dXJuIHcKICAgIHJlbj17IlRpbWUiOiJ0aW1lX2RlbHRhIiwiQWlyVGVtcCI6ImFpcl90ZW1wIiwiVHJhY2tUZW1wIjoidHJhY2tfdGVtcCIsIkh1bWlkaXR5IjoiaHVtaWRpdHkiLCJQcmVzc3VyZSI6InByZXNzdXJlIiwiUmFpbmZhbGwiOiJyYWluZmFsbCIsIldpbmREaXJlY3Rpb24iOiJ3aW5kX2RpcmVjdGlvbiIsIldpbmRTcGVlZCI6IndpbmRfc3BlZWQifQogICAgdz13LnJlbmFtZShjb2x1bW5zPXtrOnYgZm9yIGssdiBpbiByZW4uaXRlbXMoKSBpZiBrIGluIHcuY29sdW1uc30pCiAgICBpZiAndGltZV9kZWx0YScgaW4gdzoKICAgICAgICBiYXNlPXBkLlRpbWVzdGFtcChzZXNzaW9uLmRhdGUsIHR6PSdVVEMnKSBpZiBwZC5UaW1lc3RhbXAoc2Vzc2lvbi5kYXRlKS50emluZm8gaXMgTm9uZSBlbHNlIHBkLlRpbWVzdGFtcChzZXNzaW9uLmRhdGUpLnR6X2NvbnZlcnQoJ1VUQycpCiAgICAgICAgd1sndGltZXN0YW1wJ109YmFzZStwZC50b190aW1lZGVsdGEody50aW1lX2RlbHRhKQogICAgcmV0dXJuIHcK", "f1pred/legacy/proxy_experiment.py": "ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNjaXB5LnN0YXRzIGltcG9ydCBzcGVhcm1hbnIKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IG5kY2dfc2NvcmUKCgpkZWYgcmFjZV9tZXRyaWNzKGRmOiBwZC5EYXRhRnJhbWUsIHNjb3JlX2NvbDogc3RyKSAtPiBkaWN0OgogICAgcm93cz1bXQogICAgZm9yIF8sZyBpbiBkZi5ncm91cGJ5KCJyYWNlSWQiKToKICAgICAgICBnPWcuc29ydF92YWx1ZXMoc2NvcmVfY29sLGFzY2VuZGluZz1GYWxzZSkuY29weSgpOyBnWyJwcmVkX3JhbmsiXT1ucC5hcmFuZ2UoMSxsZW4oZykrMSkKICAgICAgICB3cj1pbnQoZy5sb2NbZy53aW5uZXI9PTEsInByZWRfcmFuayJdLmlsb2NbMF0pCiAgICAgICAgcmhvPXNwZWFybWFucihnLnByZWRfcmFuayxnLmZpbmlzaF9wb3NpdGlvbixuYW5fcG9saWN5PSJvbWl0Iikuc3RhdGlzdGljCiAgICAgICAgbWFlPWZsb2F0KG5wLm1lYW4obnAuYWJzKGcucHJlZF9yYW5rLWcuZmluaXNoX3Bvc2l0aW9uKSkpCiAgICAgICAgbmQ9ZmxvYXQobmRjZ19zY29yZShbZy5yZWxldmFuY2UudG9fbnVtcHkoKV0sW2dbc2NvcmVfY29sXS50b19udW1weSgpXSxrPW1pbig1LGxlbihnKSkpKQogICAgICAgIHJvd3MuYXBwZW5kKFt3cj09MSx3cjw9MyxyaG8sbWFlLG5kXSkKICAgIGE9bnAuYXNhcnJheShyb3dzLGZsb2F0KQogICAgcmV0dXJuIHsibl9yYWNlcyI6bGVuKGEpLCJ0b3AxIjphWzosMF0ubWVhbigpLCJ3aW5uZXJfdG9wMyI6YVs6LDFdLm1lYW4oKSwic3BlYXJtYW4iOm5wLm5hbm1lYW4oYVs6LDJdKSwicmFua19tYWUiOmFbOiwzXS5tZWFuKCksIm5kY2c1IjphWzosNF0ubWVhbigpfQoKCmRlZiBydW5fdXBsb2FkZWRfcmVzdWx0c19zdW1tYXJ5KGhvbGRvdXRfY3N2OiBzdHIsIGZlYXR1cmVfZXhwZXJpbWVudHNfY3N2OiBzdHIpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlByZXNlcnZlIGVtcGlyaWNhbCBWMSByZXN1bHRzIHdoaWxlIFYyIHJhdyB0ZWxlbWV0cnkgaXMgcmVydW4gb24gS2FnZ2xlLiIiIgogICAgaD1wZC5yZWFkX2Nzdihob2xkb3V0X2Nzdik7IGZlPXBkLnJlYWRfY3N2KGZlYXR1cmVfZXhwZXJpbWVudHNfY3N2KQogICAgaFsiZ3JpZF9zY29yZSJdPS1wZC50b19udW1lcmljKGguZ3JpZF9lZmZlY3RpdmUsZXJyb3JzPSJjb2VyY2UiKS5maWxsbmEoOTkpCiAgICBiYXNlbGluZT17ImZlYXR1cmVfc2V0IjoiR1JJRF9CQVNFTElORV9SRUNPTVBVVEVEIiwqKnJhY2VfbWV0cmljcyhoLCJncmlkX3Njb3JlIil9CiAgICBjb2xzPVtjIGZvciBjIGluIFsiZmVhdHVyZV9zZXQiLCJuX3JhY2VzIiwidG9wMSIsIndpbm5lcl90b3AzIiwic3BlYXJtYW4iLCJyYW5rX21hZSIsIm5kY2c1Iiwibl9mZWF0dXJlcyJdIGlmIGMgaW4gZmVdCiAgICByZXR1cm4gcGQuY29uY2F0KFtwZC5EYXRhRnJhbWUoW2Jhc2VsaW5lXSksZmVbY29sc11dLGlnbm9yZV9pbmRleD1UcnVlLHNvcnQ9RmFsc2UpCgoKZGVmIHBhaXJlZF9ib290c3RyYXBfdG9wMShkZjogcGQuRGF0YUZyYW1lLCBzY29yZV9hOiBzdHIsIHNjb3JlX2I6IHN0ciwgbl9ib290OiBpbnQ9NTAwMCwgc2VlZDogaW50PTQyKSAtPiBkaWN0OgogICAgYnk9W10KICAgIGZvciByaWQsZyBpbiBkZi5ncm91cGJ5KCJyYWNlSWQiKToKICAgICAgICB3YT1nLmxvY1tnW3Njb3JlX2FdLmlkeG1heCgpLCJ3aW5uZXIiXT09MTsgd2I9Zy5sb2NbZ1tzY29yZV9iXS5pZHhtYXgoKSwid2lubmVyIl09PTEKICAgICAgICBieS5hcHBlbmQoKGludCh3YSksaW50KHdiKSkpCiAgICBhcnI9bnAuYXNhcnJheShieSk7IGRlbHRhPShhcnJbOiwxXS1hcnJbOiwwXSkuYXN0eXBlKGZsb2F0KQogICAgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKTsgYm9vdHM9W10KICAgIGZvciBfIGluIHJhbmdlKG5fYm9vdCk6IGJvb3RzLmFwcGVuZChybmcuY2hvaWNlKGRlbHRhLHNpemU9bGVuKGRlbHRhKSxyZXBsYWNlPVRydWUpLm1lYW4oKSkKICAgIGxvLGhpPW5wLnBlcmNlbnRpbGUoYm9vdHMsWzIuNSw5Ny41XSkKICAgIHJldHVybiB7ImRlbHRhIjpmbG9hdChkZWx0YS5tZWFuKCkpLCJjaV9sb3ciOmZsb2F0KGxvKSwiY2lfaGlnaCI6ZmxvYXQoaGkpLCJuX3JhY2VzIjpsZW4oZGVsdGEpfQo=", "f1pred/legacy/__init__.py": "ZnJvbSAucGh5c2ljc19mZWF0dXJlcyBpbXBvcnQgKgpmcm9tIC5sZWFrYWdlIGltcG9ydCAqCmZyb20gLm1vZGVsaW5nIGltcG9ydCAqCg==", "tests/test_modules_pytest.py": "aW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXSkpCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgcHl0ZXN0Cgpmcm9tIHNyYy5waHlzaWNzX2ZlYXR1cmVzIGltcG9ydCAqCmZyb20gc3JjLmxlYWthZ2UgaW1wb3J0ICoKZnJvbSBzcmMubW9kZWxpbmcgaW1wb3J0ICoKZnJvbSBzcmMuZGF0YV9zb3VyY2VzIGltcG9ydCAqCgoKZGVmIHRlc3RfMDFfY3VydmF0dXJlX2NpcmNsZSgpOgogICAgdGg9bnAubGluc3BhY2UoMCwyKm5wLnBpLDUwMCk7IHI9MTAwLjAKICAgIGs9Y3VydmF0dXJlX2Zyb21feHkocipucC5jb3ModGgpLHIqbnAuc2luKHRoKSkKICAgIGFzc2VydCBhYnMobnAubmFubWVkaWFuKGtbMTA6LTEwXSktMS9yKTw1ZS00CgpkZWYgdGVzdF8wMl9jdXJ2YXR1cmVfaW5kaWNlcygpOgogICAgeD1jdXJ2YXR1cmVfaW5kaWNlcyhbMC4wMSwwLjAyLDAuMDMsMC4wNF0pCiAgICBhc3NlcnQgeFsnY3VydmF0dXJlX3NldmVyaXR5J10+eFsnY3VydmF0dXJlX2V4cG9zdXJlJ10+MAoKZGVmIHRlc3RfMDNfc2VnbWVudF9jb3JuZXJzKCk6CiAgICB0ZWw9cGQuRGF0YUZyYW1lKHsnZGlzdGFuY2UnOm5wLmFyYW5nZSgwLDEwMDEsMTApLCdzcGVlZCc6MjAwfSkKICAgIG1rPXBkLkRhdGFGcmFtZSh7J2Nvcm5lcic6WzEsMl0sJ2Rpc3RhbmNlJzpbMjAwLDcwMF19KQogICAgb3V0PXNlZ21lbnRfY29ybmVycyh0ZWwsbWssNTAsNTApCiAgICBhc3NlcnQgc2V0KG91dC5jb3JuZXIpPT17MSwyfSBhbmQgb3V0LnJlbGF0aXZlX2Rpc3RhbmNlLmFicygpLm1heCgpPD01MAoKZGVmIHRlc3RfMDRfYnJha2luZ19lZmZpY2llbmN5KCk6CiAgICBhPWJyYWtpbmdfZWZmaWNpZW5jeSgzMDAsMTAwLDEwMCkKICAgIGFzc2VydCAyMDxhPDQwCgpkZWYgdGVzdF8wNV90cmFjdGlvbl9pbmRleCgpOgogICAgYT10cmFjdGlvbl9pbmRleCgxMDAsMTcyLDIuMCkKICAgIGFzc2VydCBhYnMoYS0xMCk8MC4xCgpkZWYgdGVzdF8wNl9leGl0X2FtcGxpZmljYXRpb24oKToKICAgIGFzc2VydCBleGl0X2FtcGxpZmljYXRpb24oMy42LDgwMCk9PXB5dGVzdC5hcHByb3goODAwLjApCgpkZWYgdGVzdF8wN19zdHJhaWdodF90aW1lX2dhaW4oKToKICAgIGFzc2VydCBzdHJhaWdodF90aW1lX2dhaW4oMjAwLDE5MCwxMDAwKT4wCgpkZWYgdGVzdF8wOF9hZXJvX2NvbW1pdG1lbnQoKToKICAgIGFzc2VydCBhZXJvX2NvbW1pdG1lbnRfaW5kZXgoMjUwLDAuMDEsMTAwKT5hZXJvX2NvbW1pdG1lbnRfaW5kZXgoMjAwLDAuMDEsNzApCgpkZWYgdGVzdF8wOV93aW5kX3Byb2plY3Rpb24oKToKICAgIGgsYz13aW5kX2NvbXBvbmVudHMoMTAsMCwwKTsgYXNzZXJ0IGg9PXB5dGVzdC5hcHByb3goMTApIGFuZCBjPT1weXRlc3QuYXBwcm94KDAsYWJzPTFlLTgpCiAgICBoLGM9d2luZF9jb21wb25lbnRzKDEwLDkwLDApOyBhc3NlcnQgYWJzKGgpPDFlLTcgYW5kIGM9PXB5dGVzdC5hcHByb3goMTApCgpkZWYgdGVzdF8xMF9jb3JuZXJfbWV0cmljcygpOgogICAgZD1ucC5saW5zcGFjZSgwLDMwMCw2MSk7IHNwZWVkPW5wLnJfW25wLmxpbnNwYWNlKDI1MCwxMDAsMzEpLG5wLmxpbnNwYWNlKDEwMCwyMjAsMzApXQogICAgdGVsPXBkLkRhdGFGcmFtZSh7J2Rpc3RhbmNlJzpkLCdzcGVlZCc6c3BlZWQsJ2JyYWtlJzpucC5yX1tucC56ZXJvcygxMCksbnAub25lcygyMSksbnAuemVyb3MoMzApXSwndGhyb3R0bGUnOjgwLCdjdXJ2YXR1cmUnOjAuMDF9KQogICAgbT1jb3JuZXJfbWV0cmljcyh0ZWwsNTAwKQogICAgYXNzZXJ0IG1bJ2VudHJ5X3NwZWVkX2ttaCddPj0yNDAgYW5kIG1bJ2FwZXhfc3BlZWRfa21oJ108PTEwNSBhbmQgbVsndHJhY3Rpb25faW5kZXhfbXMyJ10+MAoKZGVmIHRlc3RfMTFfdHJhY2tfZGVtYW5kX3ZlY3RvcigpOgogICAgYz1wZC5EYXRhRnJhbWUoeydhcGV4X3NwZWVkX2ttaCc6WzEwMCwxODAsMjQwLDI1MF0sJ2JyYWtpbmdfZWZmaWNpZW5jeV9tczInOlsxMCw4LDQsM10sJ3RyYWN0aW9uX2luZGV4X21zMic6WzUsNCwyLDJdLCdhZXJvX2NvbW1pdG1lbnQnOlsuMiwuNSwxLjIsMS40XSwnZXhpdF9hbXBsaWZpY2F0aW9uJzpbMTAsMjAsMzAsNDBdfSkKICAgIHY9dHJhY2tfZGVtYW5kX3ZlY3RvcihjKTsgYXNzZXJ0IHYubG93X3NwZWVkX3NoYXJlPT1weXRlc3QuYXBwcm94KC4yNSkgYW5kIHYuaGlnaF9zcGVlZF9zaGFyZT09cHl0ZXN0LmFwcHJveCguNSkKCmRlZiB0ZXN0XzEyX3Nocmlua19tZWFuKCk6CiAgICBhc3NlcnQgc2hyaW5rX21lYW4oWzEwXSwwLHN0cmVuZ3RoPTEpPT1weXRlc3QuYXBwcm94KDUpCgpkZWYgdGVzdF8xM19jb3NpbmVfY29tcGF0aWJpbGl0eSgpOgogICAgYXNzZXJ0IGNvc2luZV9jb21wYXRpYmlsaXR5KFsxLDBdLFsxLDBdKT09cHl0ZXN0LmFwcHJveCgxKQogICAgYXNzZXJ0IGFicyhjb3NpbmVfY29tcGF0aWJpbGl0eShbMSwwXSxbMCwxXSkpPDFlLTkKCmRlZiB0ZXN0XzE0X25lYXJlc3RfdHJhY2tzKCk6CiAgICBoaXN0PXBkLkRhdGFGcmFtZSh7J2EnOlsxLDAsLjldLCdiJzpbMCwxLC4xXX0pCiAgICBuPW5lYXJlc3RfdHJhY2tzKHBkLlNlcmllcyh7J2EnOjEsJ2InOjB9KSxoaXN0LFsnYScsJ2InXSwyKQogICAgYXNzZXJ0IG4uaWxvY1swXVsnaW5kZXgnXT09MAoKZGVmIHRlc3RfMTVfdGVhbW1hdGVfYWRqdXN0ZWRfZGVjb21wb3NpdGlvbigpOgogICAgcm93cz1bXQogICAgZm9yIGNhcixiYXNlIGluIFsoJ0EnLDEpLCgnQicsLTEpXToKICAgICAgICBmb3IgZHJ2LHNraWxsIGluIFsoY2FyKycxJywuNSksKGNhcisnMicsLS41KV06CiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKDEwKTogcm93cy5hcHBlbmQoeydjYXInOmNhciwnZHJpdmVyJzpkcnYsJ3BlcmZvcm1hbmNlJzpiYXNlK3NraWxsfSkKICAgIGRyLGNhPXRlYW1tYXRlX2FkanVzdGVkX2RlY29tcG9zaXRpb24ocGQuRGF0YUZyYW1lKHJvd3MpLGFscGhhPS4xKQogICAgYXNzZXJ0IGxlbihkcik9PTQgYW5kIGxlbihjYSk9PTIKCmRlZiB0ZXN0XzE2X3RlbXBlcmF0dXJlX3NlbnNpdGl2aXR5KCk6CiAgICB0PW5wLmxpbnNwYWNlKDIwLDUwLDUwKTsgZD1wZC5EYXRhRnJhbWUoeyd0cmFja190ZW1wJzp0LCdwYWNlX3Jlc2lkdWFsJzouMDMqdCtucC5yYW5kb20uZGVmYXVsdF9ybmcoMSkubm9ybWFsKDAsLjAxLGxlbih0KSl9KQogICAgYXNzZXJ0IGFicyh0ZW1wZXJhdHVyZV9zZW5zaXRpdml0eShkKS0uMDMpPC4wMQoKZGVmIHRlc3RfMTdfd2V0X3NraWxsX3Jlc2lkdWFsKCk6CiAgICBkPXBkLkRhdGFGcmFtZSh7J2RyaXZlcic6WydBJywnQicsJ0EnLCdCJ10sJ2Nhcic6WydYJ10qNCwncGFjZV9yZXNpZHVhbCc6Wy0uMiwuMiwtLjEsLjFdLCd3ZXQnOlsxLDEsMSwxXX0pCiAgICBzPXdldF9za2lsbF9yZXNpZHVhbChkKTsgYXNzZXJ0IHNbJ0EnXT5zWydCJ10KCmRlZiB0ZXN0XzE4X3R5cmVfZGVncmFkYXRpb25fd2l0aF90ZW1wX2FuZF9taXNzaW5nKCk6CiAgICBhZ2U9bnAuYXJhbmdlKDEsNDEpOyB0ZW1wPW5wLnJhbmRvbS5kZWZhdWx0X3JuZygzKS51bmlmb3JtKDI1LDQ1LDQwKTsgeT05MCsuMDUqYWdlKy4wMDIqYWdlKioyKy4wMSp0ZW1wCiAgICBkPXBkLkRhdGFGcmFtZSh7J3R5cmVfYWdlJzphZ2UsJ2xhcF90aW1lJzp5LCd0cmFja190ZW1wJzp0ZW1wfSk7IGQubG9jWzMsJ2xhcF90aW1lJ109bnAubmFuCiAgICByPXR5cmVfZGVncmFkYXRpb24oZCk7IGFzc2VydCBhYnMoclsnbGluZWFyX2RlZyddLS4wNSk8LjAzIGFuZCBucC5pc2Zpbml0ZShyWyd0ZW1wX2VmZmVjdCddKQoKZGVmIHRlc3RfMTlfdHlyZV93YXJtdXAoKToKICAgIGFzc2VydCB0eXJlX3dhcm11cF9sYXBzKFs5Miw5MSw5MC4zLDkwLjEsOTAuMF0sLjI1KT09MwoKZGVmIHRlc3RfMjBfZnVlbF9jb3JyZWN0aW9uKCk6CiAgICBsYXBzPW5wLmFyYW5nZSgxLDYpOyBvYnNlcnZlZD05MC0uMDM1KihsYXBzLTEpCiAgICBjb3JyPWZ1ZWxfY29ycmVjdChvYnNlcnZlZCxsYXBzLC0uMDM1KQogICAgYXNzZXJ0IG5wLnN0ZChjb3JyKTwxZS04CgpkZWYgdGVzdF8yMV90cmFja19ldm9sdXRpb24oKToKICAgIGQ9cGQuRGF0YUZyYW1lKHsnc2Vzc2lvbl9zZWNvbmRzJzpucC5hcmFuZ2UoMTAwKSwnbGFwX3RpbWUnOm5wLnJfW25wLnJlcGVhdCg5Mi4sNTApLG5wLnJlcGVhdCg5MC4sNTApXX0pCiAgICBjPXRyYWNrX2V2b2x1dGlvbl9jb3JyZWN0KGQsYmlucz0yKQogICAgYXNzZXJ0IGFicyhjLmRyb3BuYSgpLmlsb2NbOjQwXS5tZWRpYW4oKS1jLmRyb3BuYSgpLmlsb2NbLTQwOl0ubWVkaWFuKCkpPC4xCgpkZWYgdGVzdF8yMl9wcmFjdGljZV9wYWNlKCk6CiAgICBkPXBkLkRhdGFGcmFtZSh7J2RyaXZlcic6WydBJ10qNitbJ0InXSo2LCdjb3JyZWN0ZWRfbGFwJzpbOTAsOTAuMSw5MC4yLDkxLDkxLjEsOTEuMiw4OSw4OS4xLDg5LjIsOTAsOTAuMSw5MC4yXSwnc3RpbnQnOlsxXSo2K1sxXSo2fSkKICAgIHA9cHJhY3RpY2VfcGFjZShkKTsgYXNzZXJ0IHAubG9jW3AuZHJpdmVyPT0nQicsJ3NpbmdsZV9sYXBfcGFjZSddLmlsb2NbMF0gPCBwLmxvY1twLmRyaXZlcj09J0EnLCdzaW5nbGVfbGFwX3BhY2UnXS5pbG9jWzBdCgpkZWYgdGVzdF8yM19kaXJ0eV9haXIoKToKICAgIGQ9cGQuRGF0YUZyYW1lKHsncGFjZV9yZXNpZHVhbCc6WzAsMC4xLC44LC45XSwnZ2FwX2FoZWFkJzpbMyw0LC41LDEuNV19KQogICAgYXNzZXJ0IGRpcnR5X2Fpcl9wZW5hbHR5KGQpPi41IGFuZCB0cmFmZmljX2NsYXNzKC41KT09J2RpcnR5X2x0MScKCmRlZiB0ZXN0XzI0X292ZXJ0YWtlX21vZGVsKCk6CiAgICBybmc9bnAucmFuZG9tLmRlZmF1bHRfcm5nKDApOyBuPTMwMDsgcGFjZT1ybmcubm9ybWFsKHNpemU9bik7IHA9MS8oMStucC5leHAoLTIqcGFjZSkpOyB5PXJuZy5yYW5kb20obik8cAogICAgZD1wZC5EYXRhRnJhbWUoeydnYXAnOnJuZy51bmlmb3JtKC4yLDIsbiksJ3BhY2VfZGVsdGEnOnBhY2UsJ3R5cmVfZGVsdGEnOnJuZy5ub3JtYWwoc2l6ZT1uKSwnc3RyYWlnaHRfbGVuZ3RoJzpybmcudW5pZm9ybSgzMDAsMTIwMCxuKSwnZHJzX2F2YWlsYWJsZSc6cm5nLmludGVnZXJzKDAsMixuKSwncGFzc2VkJzp5LmFzdHlwZShpbnQpfSkKICAgIG0sZj1maXRfb3ZlcnRha2VfbW9kZWwoZCk7IGFzc2VydCBtLmNvZWZfWzBdW2YuaW5kZXgoJ3BhY2VfZGVsdGEnKV0+MAoKZGVmIHRlc3RfMjVfcmVsaWFiaWxpdHlfcHJpb3JfYW5kX3NoaWZ0KCk6CiAgICBkPXBkLkRhdGFGcmFtZSh7J2Nhcic6WydBJ10qNSwnZG5mJzpbMSwwLDAsMCwwXX0pCiAgICByPXJlbGlhYmlsaXR5X2ZlYXR1cmVzKGQsd2luZG93PTMpOyBhc3NlcnQgci5pbG9jWzBdIT1yLmlsb2NbMF0gYW5kIDA8ci5pbG9jWy0xXTwxCgpkZWYgdGVzdF8yNl9zdHJhdGVneV9mZWF0dXJlcygpOgogICAgYXNzZXJ0IHBpdF9sb3NzKDEwMCwxMzAsOCk9PTIyCiAgICBhc3NlcnQgdW5kZXJjdXRfcG93ZXIoOTEsOTAsMjAsMik9PXB5dGVzdC5hcHByb3goLjEpCgpkZWYgdGVzdF8yN19iYWNrd2FyZF93ZWF0aGVyX2FuZF9yZXNhbXBsZSgpOgogICAgdGVsPXBkLkRhdGFGcmFtZSh7J3RpbWVzdGFtcCc6cGQudG9fZGF0ZXRpbWUoWycyMDI2LTAxLTAxVDAwOjAxWicsJzIwMjYtMDEtMDFUMDA6MDNaJ10pLCdkaXN0YW5jZSc6WzAsMTBdLCdzcGVlZCc6WzEwMCwxMjBdfSkKICAgIHc9cGQuRGF0YUZyYW1lKHsndGltZXN0YW1wJzpwZC50b19kYXRldGltZShbJzIwMjYtMDEtMDFUMDA6MDBaJywnMjAyNi0wMS0wMVQwMDowMlonXSksJ3RyYWNrX3RlbXAnOlszMCwzMV19KQogICAgb3V0PWJhY2t3YXJkX2Fzb2Zfd2VhdGhlcih0ZWwsdyk7IGFzc2VydCBsaXN0KG91dC50cmFja190ZW1wKT09WzMwLDMxXQogICAgcnI9cmVzYW1wbGVfdGVsZW1ldHJ5KHBkLkRhdGFGcmFtZSh7J2Rpc3RhbmNlJzpbMCwxMF0sJ3NwZWVkJzpbMTAwLDEyMF19KSw1KTsgYXNzZXJ0IGxpc3QocnIuc3BlZWQpPT1bMTAwLDExMCwxMjBdCgpkZWYgdGVzdF8yOF9sZWFrYWdlX3N0YWdlX2d1YXJkKCk6CiAgICBhc3NlcnQgYWxsb3dlZCgnZnAyJywnUE9TVF9GUDMnKSBhbmQgbm90IGFsbG93ZWQoJ3F1YWxpZnlpbmcnLCdQT1NUX0ZQMycpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvcik6IGFzc2VydF9ub19mdXR1cmVfZmVhdHVyZXMoWydoaXN0b3JpY2FsX2Zvcm0nLCdyYWNlX3dlYXRoZXInXSwnUE9TVF9RVUFMSScpCgpkZWYgdGVzdF8yOV90aW1lc3RhbXBfY3V0b2ZmKCk6CiAgICBnb29kPXBkLkRhdGFGcmFtZSh7J2YnOlsnMjAyNi0wMS0wMVQxMDowMFonXSwncCc6WycyMDI2LTAxLTAxVDExOjAwWiddfSk7IGFzc2VydCBhc3NlcnRfdGltZXN0YW1wX2N1dG9mZihnb29kLCdmJywncCcpCiAgICBiYWQ9cGQuRGF0YUZyYW1lKHsnZic6WycyMDI2LTAxLTAxVDEyOjAwWiddLCdwJzpbJzIwMjYtMDEtMDFUMTE6MDBaJ119KQogICAgd2l0aCBweXRlc3QucmFpc2VzKFZhbHVlRXJyb3IpOiBhc3NlcnRfdGltZXN0YW1wX2N1dG9mZihiYWQsJ2YnLCdwJykKCmRlZiB0ZXN0XzMwX3NvdXJjZXNfYW5kX3NpbXVsYXRvcigpOgogICAgdT1vcGVuZjFfdXJsKCdjYXJfZGF0YScsc2Vzc2lvbl9rZXk9MTIzLGRyaXZlcl9udW1iZXI9NCk7IGFzc2VydCAnc2Vzc2lvbl9rZXk9MTIzJyBpbiB1IGFuZCAnZHJpdmVyX251bWJlcj00JyBpbiB1CiAgICBkPXBkLkRhdGFGcmFtZSh7J2RyaXZlcic6WydBJywnQicsJ0MnXSwncGFjZV9zY29yZSc6WzIsMSwwXSwnZG5mX3Byb2InOlswLDAsMF0sJ3N0cmF0ZWd5X3NkJzpbMCwwLDBdfSkKICAgIG91dD1tb250ZV9jYXJsb19yYWNlKGQsNTAwLHNlZWQ9MSk7IGFzc2VydCBvdXQubG9jWzAsJ3dpbl9wcm9iYWJpbGl0eSddPT0xIGFuZCBhYnMob3V0Lndpbl9wcm9iYWJpbGl0eS5zdW0oKS0xKTwxZS05Cg==", "requirements.txt": "cGFuZGFzPj0yLjIKbnVtcHk+PTEuMjYKc2NpcHk+PTEuMTMKc2Npa2l0LWxlYXJuPj0xLjUKbWF0cGxvdGxpYj49My44CnhnYm9vc3Q+PTMuMApmYXN0ZjE+PTMuNgpyZXF1ZXN0cz49Mi4zMgpweWFycm93Pj0xNgpweXRlc3Q+PTguMApqb2JsaWI+PTEuNAo="}
for rel, b64 in payload.items():
    out = WORK / rel
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_bytes(base64.b64decode(b64))

sys.path.insert(0, str(WORK))
print('Embedded source written to', WORK)

## 1. Install runtime dependencies

In [ ]:
# Uncomment/re-run if Kaggle image is missing a dependency.
# %pip install -q "fastf1>=3.6" "xgboost>=3.0" pytest pyarrow requests

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

## 2. Run the 30-test validation suite

These tests validate formulas, missing-value handling, leakage guards, data-source normalization, telemetry/weather alignment and Monte Carlo consistency before any expensive downloads.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(WORK/'tests/test_modules_pytest.py')], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Validation failed')

## 3. Formula / feature catalog

In [ ]:
formula_catalog = pd.DataFrame([
    ('Curvature','XY finite-difference curvature','PRE_WEEKEND/SESSION'),
    ('Braking efficiency','(v_entry²-v_apex²)/(2*d_brake)','POST_FP1+'),
    ('Traction index','(v_exit-v_apex)/Δt','POST_FP1+'),
    ('Exit amplification','Δv_exit × downstream straight','POST_FP1+'),
    ('Aero commitment','lateral-g × throttle fraction','POST_FP1+'),
    ('Track–car compatibility','cosine(track demand, car capability)','PRE_WEEKEND'),
    ('Corner wind','wind projected onto track heading','PRE_WEEKEND/SESSION'),
    ('Tyre degradation','robust lap-time ~ age + age² + temperature','PRE_WEEKEND/SESSION'),
    ('Track evolution','time-bin pace normalization','POST_FP1+'),
    ('Dirty-air penalty','pace(dirty) - pace(clean)','PRE_WEEKEND/SESSION'),
    ('Reliability','shifted beta-smoothed DNF prior + classifier','PRE_WEEKEND'),
    ('Simulation','pace + DNF + strategy uncertainty','RACE_START'),
], columns=['family','formula/method','earliest_stage'])
formula_catalog

## 4. FastF1 public-data cache

Run this once with internet enabled. Kaggle will reuse the cache on later cells in the same run.

In [ ]:
from f1pred.legacy.session_pipeline import fastf1_session, session_corner_dataset, session_practice_summary, fastf1_weather
CACHE = WORK/'cache'/'fastf1'
CACHE.mkdir(parents=True, exist_ok=True)

# Example configuration. Change freely.
YEAR = 2025
EVENT = 'British Grand Prix'
SESSION = 'FP2'
RUN_SINGLE_SESSION = False  # set True to download/build the real session now

if RUN_SINGLE_SESSION:
    session = fastf1_session(YEAR, EVENT, SESSION, CACHE)
    corners = session_corner_dataset(session)
    practice = session_practice_summary(session)
    weather = fastf1_weather(session)
    display(corners.head())
    display(practice.head())
    display(weather.head())

## 5. Build driver corner profiles and car/track compatibility

In [ ]:
from f1pred.legacy.aggregation import aggregate_driver_corners, build_track_vector

if RUN_SINGLE_SESSION:
    driver_corner = aggregate_driver_corners(corners)
    track_vector = build_track_vector(corners)
    display(driver_corner)
    display(track_vector.to_frame('value'))

## 6. Leakage-safe archived forecast URLs

For historical backtests, do **not** use realized race weather as a pre-race feature. Use a forecast run that existed before your prediction timestamp. Open-Meteo Previous Runs is included for this purpose.

In [ ]:
from f1pred.legacy.data_sources import openmeteo_previous_url

# Example only: supply the circuit coordinates/date from your circuits table.
example_url = openmeteo_previous_url(
    lat=52.0786, lon=-1.0169,
    start_date='2025-07-06', end_date='2025-07-06',
    hourly=['temperature_2m','precipitation_probability','wind_speed_10m','wind_direction_10m']
)
print(example_url)

## 7. Multi-session feature cache

This helper materializes FP1/FP2/FP3/Q corner and practice features. Store these Parquet files once, then train repeatedly without re-downloading telemetry.

In [ ]:
from pathlib import Path

def cache_event(year, event, sessions=('FP1','FP2','FP3','Q')):
    outdir = WORK/'data'/'raw_features'/str(year)/str(event).replace(' ','_')
    outdir.mkdir(parents=True, exist_ok=True)
    summary=[]
    for sc in sessions:
        try:
            s = fastf1_session(year, event, sc, CACHE)
            c = session_corner_dataset(s)
            p = session_practice_summary(s) if sc.startswith('FP') else pd.DataFrame()
            w = fastf1_weather(s)
            c.to_parquet(outdir/f'{sc}_corners.parquet', index=False)
            if len(p): p.to_parquet(outdir/f'{sc}_practice.parquet', index=False)
            if len(w): w.to_parquet(outdir/f'{sc}_weather.parquet', index=False)
            summary.append((sc,len(c),len(p),len(w)))
        except Exception as e:
            summary.append((sc,0,0,0,str(e)))
    return pd.DataFrame(summary)

# Example:
# cache_event(2025, 'British Grand Prix')

## 8. Train grouped race ranker + separate DNF model

Use an expanding chronological split. One race is one LambdaMART query (`qid`). Do not random-split driver rows across future/past races.

In [ ]:
from f1pred.legacy.modeling import fit_xgb_ranker, fit_dnf_classifier, race_ranking_metrics, temperature_calibrate, softmax_by_race

# Expected engineered table schema:
# raceId, driverId, year, finish_position, winner, relevance, + pre-race features
#
# Example:
# train = model_df[model_df.year < 2025]
# test  = model_df[model_df.year == 2025]
# model, pred = fit_xgb_ranker(train, test, FEATURES)
# print(race_ranking_metrics(pred, 'score'))

## 9. Monte Carlo race outcomes

In [ ]:
from f1pred.legacy.modeling import monte_carlo_race

demo = pd.DataFrame({
    'driver':['A','B','C','D'],
    'pace_score':[1.3,1.0,.4,0.0],
    'dnf_prob':[.05,.06,.08,.10],
    'strategy_sd':[.18,.18,.22,.25],
})
display(monte_carlo_race(demo, n_sims=20000, seed=42)[['driver','win_probability','podium_probability','dnf_probability_sim']])

## 10. Recommended chronological ablation

Run these in order and keep the model family fixed first:

1. grid baseline
2. + qualifying gap/telemetry
3. + corrected FP1/FP2/FP3 pace
4. + corner braking/traction/aero
5. + track–car compatibility
6. + teammate-adjusted driver corner skill
7. + archived weather + corner wind
8. + tyre degradation/warm-up priors
9. + dirty-air/overtake priors
10. + reliability/component state
11. + strategy uncertainty
12. Monte Carlo outcome layer

Report 2025 and 2026 separately; 2026 is a regulation-regime stress test.

## 11. Export everything

In [ ]:
# Put your produced CSV/Parquet/plots under WORK/results, WORK/data and WORK/images.
archive = shutil.make_archive('/kaggle/working/F1_Predictor_V2_RESULTS' if Path('/kaggle/working').exists() else 'F1_Predictor_V2_RESULTS', 'zip', WORK)
print('Created:', archive)